# Step 13 — assign final block IDs to selected child blocks

**# of cells in notebook:** 1

**Purpose:** Assign a final `block_id` to each selected child block using the parent block identifier, the child subdivision identifier, the original LargePop and heterogeneity screening status, and whether the selected subdivision was produced by the BEAM workflow.

**Input:**

- `heterogeneous_largePop_selection`
- within each selected block folder:
  - `new_blocks_populated.gpkg`
- `heterogeneous_largePop_blocks` from Step 1, containing:
  - `block_id`
  - `LargePop`
  - `HH_CC`
  - `HH_Grtr10ha`
  - `CC_Grtr10ha`
- `beam_selection_summary.csv` from Step 12, used to identify parent blocks whose canonical selection was replaced by a BEAM result

**Output:**

- updated `new_blocks_populated.gpkg` files with final child `block_id` values
- optional `.bak` copies of the pre-update GeoPackages

Final IDs follow:

`blk_<parent block>_<subdivision>_<LargePop>_<heterogeneity>_<beam>`

where the heterogeneity code is:

- `0` — none of the heterogeneity flags
- `1` — `HH_CC`
- `2` — `HH_Grtr10ha`
- `3` — `CC_Grtr10ha`

and the final BEAM indicator is:

- `0` — standard clustering-derived subdivision
- `1` — BEAM-derived subdivision

The notebook does **not** add citywide `was_split` or `beam` attribute columns. Those binary fields are introduced later in Step 15, where all citywide blocks—split and unsplit—can be classified consistently.

**Main logic:**

**Cell 1 — Recalculate final child block IDs**

1. Builds a lookup of source-block LargePop and heterogeneity status.
2. Reads `beam_selection_summary.csv` and identifies parent blocks with a successfully selected BEAM solution.
3. Iterates through the selected block folders and reads each canonical child-block GeoPackage.
4. Identifies the subdivision number from `cluster_smooth` for standard selections or the available `part_n*` field for BEAM selections.
5. Uses the Step 12 BEAM-selection summary as the authoritative BEAM indicator, with the subdivision-field type as a fallback when the summary is unavailable.
6. Constructs the final `block_id` using parent block, subdivision, LargePop status, heterogeneity code, and BEAM indicator.
7. Rewrites each GeoPackage while preserving geometry and all other attributes.
8. Optionally creates a backup before replacing the original file and reports the number of blocks, layers, and child features updated.


In [ ]:

"""
Assign final block IDs to selected child blocks.

For every block folder under BASE_DIR, the script opens:
    <block_folder>/new_blocks_populated.gpkg

Subdivision identifiers are read from:
    - cluster_smooth for standard clustering-derived selections, or
    - the available part_n* field for BEAM-derived selections.

Final block IDs use:
    blk_{parent_block}_{subdivision}_{LargePop}_{heterogeneous}_{beam}

BEAM lineage is read from beam_selection_summary.csv when available. The
subdivision-field type is retained as a fallback for older/incomplete runs.

This step does not add was_split or beam attribute columns. Those citywide
lineage fields are created in Step 15.

The script rewrites each GeoPackage through a temporary file and can preserve
a .bak copy of the original.
"""

from __future__ import annotations

import os
import re
import shutil
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import geopandas as gpd
import pandas as pd
import pyogrio


# ---------------------------------------------------------------------
# User inputs
# ---------------------------------------------------------------------

BASE_DIR = Path(r"E:\_johannesburg\_analysis\heterogeneous_largePop_selection")

LOOKUP_GDB = Path(r"E:\_johannesburg\_analysis\blocks\blocks.gdb")
LOOKUP_LAYER = "heterogeneous_largePop_blocks"

GPKG_NAME = "new_blocks_populated.gpkg"
BEAM_SELECTION_SUMMARY = BASE_DIR / "beam_selection_summary.csv"

MAKE_BACKUPS = True
WRITE_CHANGES = True

TARGET_FIELD = "block_id"
STANDARD_SUBDIVISION_FIELD = "cluster_smooth"

LOOKUP_BLOCK_FIELD = "block_id"
LARGEPOP_FIELD = "LargePop"
HH_CC_FIELD = "HH_CC"
HH_GRTR10HA_FIELD = "HH_Grtr10ha"
CC_GRTR10HA_FIELD = "CC_Grtr10ha"


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def normalize_block_id(value) -> Optional[str]:
    """Return the numeric parent-block ID as a string."""
    if value is None or pd.isna(value):
        return None

    text = str(value).strip()

    if re.fullmatch(r"\d+\.0", text):
        return text[:-2]
    if re.fullmatch(r"\d+", text):
        return text

    matches = re.findall(r"\d+", text)
    if not matches:
        return None

    if text.startswith("blk_") and len(matches) >= 2:
        return matches[1]

    return matches[0]


def numeric_like(value, default: int = 0) -> int:
    if value is None or pd.isna(value):
        return default
    try:
        return int(float(value))
    except Exception:
        return default


def clean_subdivision_value(value, field_name: str) -> str:
    if value is None or pd.isna(value):
        raise ValueError(f"{field_name} contains a missing value")

    try:
        numeric = float(value)
        if numeric.is_integer():
            return str(int(numeric))
    except Exception:
        pass

    text = str(value).strip()
    if not text:
        raise ValueError(f"{field_name} contains a blank value")
    return text


def find_field_case_insensitive(fields: Iterable[str], desired: str) -> Optional[str]:
    lookup = {f.lower(): f for f in fields}
    return lookup.get(desired.lower())


def find_subdivision_field(fields: Iterable[str]) -> Tuple[str, int]:
    """
    Return (field_name, inferred_beam).

    Standard selections use cluster_smooth.
    BEAM selections use one part_n* field.
    """
    fields = list(fields)

    standard = find_field_case_insensitive(fields, STANDARD_SUBDIVISION_FIELD)
    if standard is not None:
        return standard, 0

    part_fields = sorted(
        [f for f in fields if re.fullmatch(r"part_n\d+", str(f), flags=re.IGNORECASE)],
        key=lambda x: int(re.findall(r"\d+", str(x))[0]),
    )

    if len(part_fields) == 1:
        return part_fields[0], 1

    if len(part_fields) > 1:
        raise RuntimeError(
            f"Multiple BEAM subdivision fields found: {part_fields}. "
            "The canonical selected layer should contain only one part_n* field."
        )

    raise RuntimeError(
        f"No subdivision field found. Expected '{STANDARD_SUBDIVISION_FIELD}' "
        "or one part_n* field."
    )


def heterogeneous_code(
    row: pd.Series,
    hh_cc_field: str,
    hh_grtr_field: str,
    cc_grtr_field: str,
) -> int:
    if numeric_like(row[hh_cc_field]) == 1:
        return 1
    if numeric_like(row[hh_grtr_field]) == 1:
        return 2
    if numeric_like(row[cc_grtr_field]) == 1:
        return 3
    return 0


def build_lookup_table() -> Dict[str, Tuple[int, int]]:
    print("Reading lookup layer...")
    print(f"  GDB:   {LOOKUP_GDB}")
    print(f"  Layer: {LOOKUP_LAYER}")

    lookup = pyogrio.read_dataframe(
        LOOKUP_GDB,
        layer=LOOKUP_LAYER,
        read_geometry=False,
    )
    fields = list(lookup.columns)

    block_field = find_field_case_insensitive(fields, LOOKUP_BLOCK_FIELD)
    largepop_field = find_field_case_insensitive(fields, LARGEPOP_FIELD)
    hh_cc_field = find_field_case_insensitive(fields, HH_CC_FIELD)
    hh_grtr_field = find_field_case_insensitive(fields, HH_GRTR10HA_FIELD)
    cc_grtr_field = find_field_case_insensitive(fields, CC_GRTR10HA_FIELD)

    required = {
        LOOKUP_BLOCK_FIELD: block_field,
        LARGEPOP_FIELD: largepop_field,
        HH_CC_FIELD: hh_cc_field,
        HH_GRTR10HA_FIELD: hh_grtr_field,
        CC_GRTR10HA_FIELD: cc_grtr_field,
    }

    missing = [label for label, actual in required.items() if actual is None]
    if missing:
        raise RuntimeError(
            "Lookup layer is missing required field(s): "
            + ", ".join(missing)
            + f"\nFields found: {fields}"
        )

    out: Dict[str, Tuple[int, int]] = {}
    duplicate_ids = []

    for _, row in lookup.iterrows():
        bid = normalize_block_id(row[block_field])
        if bid is None:
            continue

        largepop = numeric_like(row[largepop_field])
        hetero = heterogeneous_code(
            row,
            hh_cc_field,
            hh_grtr_field,
            cc_grtr_field,
        )

        if bid in out:
            duplicate_ids.append(bid)

        out[bid] = (largepop, hetero)

    if duplicate_ids:
        unique_dupes = sorted(
            set(duplicate_ids),
            key=lambda x: int(x) if x.isdigit() else x,
        )
        print(
            "WARNING: duplicate lookup block_id values found; "
            f"last value used: {unique_dupes[:20]}"
        )

    print(f"Lookup records loaded: {len(out):,}")
    return out


def load_selected_beam_parents() -> tuple[set[str], bool]:
    """
    Return parent block IDs whose Step 12 BEAM selection status is "selected".

    The second return value indicates whether the summary file was available.
    """
    if not BEAM_SELECTION_SUMMARY.exists():
        print(
            "WARNING: BEAM selection summary not found. "
            "BEAM lineage will be inferred from subdivision fields."
        )
        print(f"  {BEAM_SELECTION_SUMMARY}")
        return set(), False

    df = pd.read_csv(BEAM_SELECTION_SUMMARY)
    required = {"status", "lookup_block_id"}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise RuntimeError(
            "BEAM selection summary is missing required column(s): "
            + ", ".join(missing)
        )

    selected = df[
        df["status"].astype(str).str.lower().eq("selected")
    ].copy()

    parent_ids = {
        bid
        for bid in (
            normalize_block_id(v)
            for v in selected["lookup_block_id"].tolist()
        )
        if bid is not None
    }

    print(f"Selected BEAM parent blocks loaded: {len(parent_ids):,}")
    return parent_ids, True


def get_block_folders(base_dir: Path) -> list[Path]:
    folders = [
        p
        for p in base_dir.iterdir()
        if p.is_dir() and re.fullmatch(r"_\d+", p.name)
    ]
    folders.sort(key=lambda p: int(p.name.lstrip("_")))
    return folders


def read_all_layers(gpkg_path: Path) -> Dict[str, gpd.GeoDataFrame]:
    layer_info = pyogrio.list_layers(gpkg_path)
    layer_names = [str(row[0]) for row in layer_info]

    return {
        layer_name: pyogrio.read_dataframe(gpkg_path, layer=layer_name)
        for layer_name in layer_names
    }


def write_all_layers_to_temp(
    layers: Dict[str, gpd.GeoDataFrame],
    temp_gpkg: Path,
) -> None:
    if temp_gpkg.exists():
        temp_gpkg.unlink()

    for layer_name, gdf in layers.items():
        pyogrio.write_dataframe(
            gdf,
            temp_gpkg,
            layer=layer_name,
            driver="GPKG",
        )


def process_one_gpkg(
    gpkg_path: Path,
    parent_block: str,
    lookup: Dict[str, Tuple[int, int]],
    selected_beam_parents: set[str],
    beam_summary_available: bool,
) -> Tuple[int, int]:
    if parent_block not in lookup:
        print(
            f"  WARNING: parent block {parent_block} was not found "
            "in lookup layer. Skipping."
        )
        return 0, 0

    largepop, hetero = lookup[parent_block]

    layers = read_all_layers(gpkg_path)
    if not layers:
        print("  WARNING: no layers found in GeoPackage. Skipping.")
        return 0, 0

    layers_updated = 0
    rows_updated = 0

    for layer_name, gdf in layers.items():
        if gdf.empty:
            print(f"  Layer {layer_name}: skipped; layer is empty.")
            continue

        fields = list(gdf.columns)
        target_field = find_field_case_insensitive(fields, TARGET_FIELD)

        if target_field is None:
            print(
                f"  Layer {layer_name}: skipped; missing {TARGET_FIELD}."
            )
            continue

        subdivision_field, inferred_beam = find_subdivision_field(fields)

        if beam_summary_available:
            beam_flag = int(parent_block in selected_beam_parents)
            if beam_flag != inferred_beam:
                print(
                    "    WARNING: BEAM summary and subdivision-field type "
                    f"disagree (summary={beam_flag}, inferred={inferred_beam}). "
                    "Using the Step 12 summary."
                )
        else:
            beam_flag = int(inferred_beam)

        subdivisions = [
            clean_subdivision_value(v, subdivision_field)
            for v in gdf[subdivision_field]
        ]

        old_unique = sorted(gdf[target_field].astype(str).unique().tolist())

        gdf[target_field] = [
            f"blk_{parent_block}_{subdivision}_{largepop}_{hetero}_{beam_flag}"
            for subdivision in subdivisions
        ]

        new_unique = sorted(gdf[target_field].astype(str).unique().tolist())

        layers[layer_name] = gdf
        layers_updated += 1
        rows_updated += len(gdf)

        print(f"  Layer {layer_name}:")
        print(f"    subdivision field: {subdivision_field}")
        print(f"    rows updated:      {len(gdf):,}")
        print(f"    beam flag in ID:   {beam_flag}")
        print(f"    old block_id values: {old_unique[:10]}")
        print(f"    new block_id values: {new_unique[:10]}")

    if layers_updated == 0:
        return 0, 0

    if not WRITE_CHANGES:
        print("  DRY RUN: WRITE_CHANGES is False, so no file was written.")
        return layers_updated, rows_updated

    temp_gpkg = gpkg_path.with_name(
        gpkg_path.stem + "__temp_write.gpkg"
    )
    backup_gpkg = gpkg_path.with_suffix(gpkg_path.suffix + ".bak")

    write_all_layers_to_temp(layers, temp_gpkg)

    if MAKE_BACKUPS and not backup_gpkg.exists():
        shutil.copy2(gpkg_path, backup_gpkg)
        print(f"  Backup written: {backup_gpkg.name}")
    elif MAKE_BACKUPS and backup_gpkg.exists():
        print(
            f"  Backup already exists, not overwritten: {backup_gpkg.name}"
        )

    os.replace(temp_gpkg, gpkg_path)
    print("  GeoPackage replaced successfully.")

    return layers_updated, rows_updated


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

def main() -> None:
    if not BASE_DIR.exists():
        raise RuntimeError(f"BASE_DIR does not exist: {BASE_DIR}")
    if not LOOKUP_GDB.exists():
        raise RuntimeError(f"LOOKUP_GDB does not exist: {LOOKUP_GDB}")

    print("\nSettings")
    print("--------")
    print(f"Base directory: {BASE_DIR}")
    print(f"GeoPackage name: {GPKG_NAME}")
    print(f"Write changes: {WRITE_CHANGES}")
    print(f"Make backups: {MAKE_BACKUPS}")

    lookup = build_lookup_table()
    selected_beam_parents, beam_summary_available = load_selected_beam_parents()
    block_folders = get_block_folders(BASE_DIR)

    print("\nProcessing block folders")
    print("------------------------")
    print(f"Block folders found: {len(block_folders):,}")

    total_blocks_processed = 0
    total_blocks_skipped = 0
    total_layers_updated = 0
    total_rows_updated = 0

    for folder in block_folders:
        parent_block = normalize_block_id(folder.name)
        assert parent_block is not None

        gpkg_path = folder / GPKG_NAME

        if not gpkg_path.exists():
            print(
                f"\nBlock {folder.name}: WARNING, "
                f"{GPKG_NAME} not found. Skipping."
            )
            total_blocks_skipped += 1
            continue

        print(f"\nBlock {folder.name}")
        print(f"  Parent block: {parent_block}")
        print(f"  GeoPackage: {gpkg_path}")

        try:
            layers_updated, rows_updated = process_one_gpkg(
                gpkg_path,
                parent_block,
                lookup,
                selected_beam_parents,
                beam_summary_available,
            )
        except Exception as exc:
            print(f"  ERROR processing {folder.name}: {exc}")
            total_blocks_skipped += 1
            continue

        if layers_updated > 0:
            total_blocks_processed += 1
            total_layers_updated += layers_updated
            total_rows_updated += rows_updated
        else:
            total_blocks_skipped += 1

    print("\nDone")
    print("----")
    print(f"Blocks processed: {total_blocks_processed:,}")
    print(f"Blocks skipped:   {total_blocks_skipped:,}")
    print(f"Layers updated:   {total_layers_updated:,}")
    print(f"Rows updated:     {total_rows_updated:,}")


if __name__ == "__main__":
    main()